In [0]:
from openai import OpenAI
import base64
import sys
import pydicom

endpoint = "https://telefonica-hackathon-2026.cognitiveservices.azure.com/openai/v1/"
model_name = "gpt-4.1-mini"
deployment_name = "gpt-4.1-mini"

api_key = "API_KEY"

import base64
import io
import numpy as np
import pydicom
from PIL import Image


def dicom_to_base64_png(dicom_path: str) -> str:
    """
    Convert a DICOM file to a PNG image in memory and return Base64 string.

    Args:
        dicom_path (str): Path to the DICOM file.

    Returns:
        str: Base64-encoded PNG image suitable for LLM input.
    """

    # Load DICOM
    ds = pydicom.dcmread(dicom_path)

    # Extract pixel array
    pixel_array = ds.pixel_array.astype(np.float32)

    # Normalize to 0-255
    pixel_array -= pixel_array.min()
    pixel_array /= pixel_array.max()
    pixel_array *= 255.0
    pixel_array = pixel_array.astype(np.uint8)

    # Convert to PIL image
    image = Image.fromarray(pixel_array)

    # Save PNG to memory buffer
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")

    # Convert to Base64
    base64_str = base64.b64encode(buffer.getvalue()).decode("utf-8")

    return base64_str



def base64_to_llm(base64_img, system_prompt, user_prompt):
  client = OpenAI(
              base_url=f"{endpoint}",
              api_key=api_key
          )

  completion = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {
          "role": "system",
          "content": system_prompt,
        },
        {
        "role": "user",
        "content": [
                  {"type": "text", "text": user_prompt},

                  {

                      "type": "image_url",

                      "image_url": {

                          "url": f"data:image/png;base64,{base64_img}"

                      }

                  }

              ]
        }
  ,
      ],
    )
  return completion

def ask_llm(png_file, system_prompt, user_prompt):
    completion = base64_to_llm(png_to_base64(png_file), system_prompt, user_prompt)
    return completion




if __name__ == "__main__":
    #png_file = sys.argv[1]
    #system_prompt = sys.argv[2]
    #user_prompt = sys.argv[3]
    dicom_file = "/Volumes/1_inland/sectra/vone/Hackathon2/000001/000001/000002.dcm"
    system_prompt = "Output the result in a structured format with 3 required fields: is_text_detected, is_personal_information_detected, and confidence_score. Valid values for is_text_detected are yes, no, and not_sure. Valid values for is_personal_information_detected are yes, no, and not_sure. confidence_score is an integer from 0 to 100. 0 indicates high confidence of no text detected. 100 indicates high confidence of text being detected. 50 indicates uncertain. Do not leave any field empty. Always respond with 3 fields everytime."
    user_prompt = "Tell me if there is any text in the image. Also tell me if these texts contain any personal information e.g. name, date of birth, scan date, ID numbers. Then give me a confidence score."

    #completion = base64_to_llm(png_to_base64(png_file), system_prompt, user_prompt)
    #print(completion.choices[0].message.content)

    completion = base64_to_llm(dicom_to_base64_png(dicom_file), system_prompt, user_prompt)
    print(completion.choices[0].message.content)